<a href="https://colab.research.google.com/github/mdrs-thiago/mdrs-thiago.github.io/blob/main/topicos_a/2027/notebooks/lab01_mlp_fashionmnist.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# 🔬 Lab 1 — MLP em PyTorch (FashionMNIST)
**Tópicos Especiais A · Deep Learning · UERJ 2027.1** · Aula 04 · Bloco 1 — Fundamentos

**Objetivo:** construir um pipeline completo e reprodutível de treino e comparar 3 configurações de otimização/regularização.

| | |
|---|---|
| ✅ **Checkpoint (em sala)** | 60 XP: veja as marcações `✅ CHECKPOINT` |
| 📦 **Entrega (7 dias)** | 180 XP: tabela das 3 configurações, curvas, análise por configuração e acurácia de teste da melhor. |

> Preencha apenas os trechos marcados com `# TODO`. Antes de começar: **Arquivo → Salvar uma cópia no Drive** e, se indicado, **Ambiente de execução → Alterar tipo → T4 GPU**.

## 0. Setup

In [ ]:
import random, numpy as np, torch
def set_seed(s=42):
    random.seed(s); np.random.seed(s); torch.manual_seed(s); torch.cuda.manual_seed_all(s)
set_seed(42)
device = 'cuda' if torch.cuda.is_available() else 'cpu'
print('device:', device, '| torch', torch.__version__)

In [ ]:
import time, torch.nn as nn, matplotlib.pyplot as plt
from torch.utils.data import DataLoader, random_split
from torchvision import datasets, transforms

## 1. Dados
FashionMNIST: 60 mil imagens de treino 28×28 (10 classes) e 10 mil de teste.
Vamos separar **5 mil imagens de treino para validação**. As médias/desvios abaixo foram calculados **só no treino** (por quê?).

In [ ]:
tfm = transforms.Compose([transforms.ToTensor(), transforms.Normalize((0.2860,), (0.3530,))])
full = datasets.FashionMNIST('data', train=True, download=True, transform=tfm)
test_ds = datasets.FashionMNIST('data', train=False, download=True, transform=tfm)
g = torch.Generator().manual_seed(42)
train_ds, val_ds = random_split(full, [55_000, 5_000], generator=g)

# TODO: crie os DataLoaders (batch_size=128; shuffle só no treino)
train_loader = ...
val_loader = ...
test_loader = ...
CLASSES = full.classes

In [ ]:
# Visualize alguns exemplos (sanidade!)
xb, yb = next(iter(train_loader))
fig, axs = plt.subplots(2, 8, figsize=(12, 3.5))
for ax, x, y in zip(axs.flat, xb, yb):
    ax.imshow(x[0], cmap='gray'); ax.set_title(CLASSES[y], fontsize=8); ax.axis('off')
print(xb.shape, yb.shape)

## 2. Modelo
Um MLP: `Flatten → Linear(784, h) → ReLU → (Dropout) → Linear(h, h) → ReLU → (Dropout) → Linear(h, 10)`.

In [ ]:
class MLP(nn.Module):
    def __init__(self, hidden=256, p_drop=0.0):
        super().__init__()
        # TODO: monte self.net com nn.Sequential conforme a descrição acima
        self.net = ...
    def forward(self, x):
        return self.net(x)

m = MLP()
print(m)
print('parâmetros:', sum(p.numel() for p in m.parameters()))

## 3. Loop de treino e de validação
Os 5 passos da Aula 2: `zero_grad` → forward → perda → `backward` → `step`.

In [ ]:
loss_fn = nn.CrossEntropyLoss()

def train_one_epoch(model, loader, opt):
    model.train()
    total, loss_sum, correct = 0, 0.0, 0
    for xb, yb in loader:
        xb, yb = xb.to(device), yb.to(device)
        # TODO: zero_grad, forward, perda, backward, step
        logits = ...
        loss = ...

        loss_sum += loss.item() * len(yb)
        correct += (logits.argmax(1) == yb).sum().item()
        total += len(yb)
    return loss_sum / total, correct / total

@torch.no_grad()
def evaluate(model, loader):
    model.eval()
    total, loss_sum, correct = 0, 0.0, 0
    for xb, yb in loader:
        xb, yb = xb.to(device), yb.to(device)
        logits = model(xb)
        loss_sum += loss_fn(logits, yb).item() * len(yb)
        correct += (logits.argmax(1) == yb).sum().item()
        total += len(yb)
    return loss_sum / total, correct / total

In [ ]:
import copy
def fit(model, opt, epochs=10, patience=3):
    """Treina com early stopping na perda de validação. Retorna histórico e o melhor modelo."""
    hist = {'train_loss': [], 'val_loss': [], 'train_acc': [], 'val_acc': []}
    best, best_state, bad, t0 = float('inf'), None, 0, time.time()
    for ep in range(epochs):
        tl, ta = train_one_epoch(model, train_loader, opt)
        vl, va = evaluate(model, val_loader)
        for k, v in zip(hist, (tl, vl, ta, va)): hist[k].append(v)
        print(f'época {ep+1:2d} | train {tl:.3f}/{ta:.3f} | val {vl:.3f}/{va:.3f}')
        if vl < best:
            best, best_state, bad = vl, copy.deepcopy(model.state_dict()), 0
        else:
            bad += 1
            if bad >= patience: print('early stopping'); break
    model.load_state_dict(best_state)
    hist['tempo_s'] = time.time() - t0
    return hist

def plot_hist(hists):
    fig, (a1, a2) = plt.subplots(1, 2, figsize=(12, 4))
    for name, h in hists.items():
        a1.plot(h['train_loss'], '--', label=f'{name} treino'); a1.plot(h['val_loss'], label=f'{name} val')
        a2.plot(h['val_acc'], label=name)
    a1.set_title('perda'); a2.set_title('acurácia de validação'); a1.legend(fontsize=7); a2.legend()
    plt.show()

### Teste de sanidade: overfit de um único batch
Se a perda não chegar perto de zero, há bug no loop (receita da Aula 3).

In [ ]:
set_seed(0)
m = MLP().to(device)
opt = torch.optim.Adam(m.parameters(), lr=1e-3)
xb, yb = next(iter(train_loader)); xb, yb = xb.to(device), yb.to(device)
for i in range(200):
    loss = loss_fn(m(xb), yb); opt.zero_grad(); loss.backward(); opt.step()
print('perda após 200 passos no mesmo batch:', loss.item())

## 4. ✅ CHECKPOINT
Treine uma configuração qualquer até **acurácia de validação ≥ 85%** e mostre as curvas ao professor.

In [ ]:
set_seed(42)
model = MLP().to(device)
opt = torch.optim.AdamW(model.parameters(), lr=1e-3)
h_base = fit(model, opt, epochs=8)
plot_hist({'checkpoint': h_base})

## 5. Experimentos controlados
Escolha **3 configurações** que testem uma hipótese clara. Mude **uma coisa por vez**. Exemplos:
SGD lr=0,1 vs. AdamW lr=1e-3 vs. AdamW + dropout 0,3 + weight decay 1e-2.

**Hipótese (escreva aqui):** ...

In [ ]:
configs = {
    # TODO: defina 3 configurações. Cada uma: (kwargs do MLP, função que cria o otimizador)
    'A': (dict(hidden=256, p_drop=0.0), lambda p: torch.optim.SGD(p, lr=0.1)),
    'B': ...,
    'C': ...,
}
results, hists, models = [], {}, {}
for name, (mkw, mk_opt) in configs.items():
    set_seed(42)
    m = MLP(**mkw).to(device)
    h = fit(m, mk_opt(m.parameters()), epochs=15)
    hists[name], models[name] = h, m
    best_ep = int(np.argmax(h['val_acc']))
    results.append({'config': name, 'melhor acc val': max(h['val_acc']), 'época': best_ep + 1, 'tempo (s)': round(h['tempo_s'], 1)})
plot_hist(hists)
import pandas as pd; pd.DataFrame(results)

## 6. Teste (uma única vez, na melhor configuração)

In [ ]:
# TODO: escolha a melhor configuração pela VALIDAÇÃO e avalie no teste
best_name = ...
print('teste:', evaluate(models[best_name], test_loader))

## 7. Análise (📦 entrega)
Para cada configuração, 1 parágrafo usando o vocabulário da Aula 3 (overfitting, underfitting, LR, regularização, convergência):

- **A:** ...
- **B:** ...
- **C:** ...

**Conclusão sobre a hipótese:** ...